# Import library

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Simple dataset

In [24]:
np.random.seed(1)
X = np.random.randn(200,2)   # 200 points, 2 features
y = (X[:,0] + X[:,1] > 0).astype(int).reshape(-1,1)
# Sum > 0 then class = 1 otherwise 0

# Activation Functions

In [25]:
# Sigmoid output 0 to 1
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Used in backdrop
def sigmoid_derivative(a):
    return a * (1 - a)

# ReLU (Remove negative values)
def relu(z):
    return np.maximum(0, z)

# Derivative of ReLU
def relu_derivative(z):
    return (z > 0).astype(float)

# TanH function  (range -1 to 1)
def tanh(z):
    return np.tanh(z)

# Derivative of tanh
def tanh_derivative(a):
    return 1 - np.square(a)

# Loss Function  (Binary Crossentropy)

In [26]:
def compute_loss(y, y_hat, weights, lambda_reg=0, reg_type="L2"):
    m = y.shape[0]
    loss = -np.mean(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8))

    # Regularization
    if reg_type == "L2":
        reg = (lambda_reg / (2 * m)) * sum([np.sum(w**2) for w in weights])
    elif reg_type == "L1":
        reg = (lambda_reg / m) * sum([np.sum(np.abs(w)) for w in weights])
    else:
        reg = 0

    return loss + reg

# Neural Network Class

In [27]:
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, activation="relu"):
        # Initialize weight with small random values
        self.W1 = np.random.randn(input_size, hidden_size) * 0.1
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, 1) * 0.1
        self.b2 = np.zeros((1, 1))

        self.activation = activation

    # Apply selected sctivation function
    def activate(self, z):
        if self.activation == "sigmoid":
            return sigmoid(z)
        elif self.activation == "tanh":
            return tanh(z)
        else:
            return relu(z)

    # Derivative of activation (used in backpropagation)
    def activate_derivative(self, z, a):
        if self.activation == "sigmoid":
            return sigmoid_derivative(a)
        elif self.activation == "tanh":
            return tanh_derivative(a)
        else:
            return relu_derivative(z)

    # Forward Propagation
    def forward(self, X):
        # Input layer to hidden layer
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self.activate(self.Z1)

        # Hidden layer to output layer
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = sigmoid(self.Z2)  # Final output (Probability)
        return self.A2

    # Back Propagation
    def backward(self, X, y, lr=0.01, lambda_reg=0, reg_type="L2"):
        m = X.shape[0]

        # Error at output layer
        dZ2 = self.A2 - y
        # Gradients for output layer
        dW2 = np.dot(self.A1.T, dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        # Error propagated to hidden layer
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self.activate_derivative(self.Z1, self.A1)

        # Gradients for hidden layer
        dW1 = np.dot(X.T, dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        # Update weights
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        self.W2 -= lr * dW2
        self.b2 -= lr * db2

    # Training Loop
    def train(self, X, y, epochs=1000, lr=0.01, lambda_reg=0, reg_type="L2"):
        losses = []
        for i in range(epochs):
            # Forward pass
            y_hat = self.forward(X)
            # Compute loss
            loss = compute_loss(y, y_hat, [self.W1, self.W2], lambda_reg, reg_type)
            losses.append(loss)

            #
            self.backward(X, y, lr, lambda_reg, reg_type)

        return losses

    # Prediction
    def predict(self, X):
        return (self.forward(X) > 0.5).astype(int)

In [28]:
# Create model
model = NeuralNetwork(input_size=2, hidden_size=8, activation="relu")

# Train model
losses = model.train(X, y, epochs=500, lr=0.01)

# Print final loss
print("Final Loss:", losses[-1])

# Check accuracy
pred = model.predict(X)
accuracy = np.mean(pred == y)
print("Accuracy:", accuracy)

Final Loss: 0.5800262349966885
Accuracy: 0.92
